# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL: 

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. We'll use the Croissant schema URL.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
from pprint import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Get metadata summary as a Python dict
metadata = json.loads(dataset.metadata.to_json())

print(f"{metadata['name']}: {metadata['description']}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s. We'll print the list of record sets, their IDs and the contained fields (column `@id`s).

In [ ]:
# List available record sets with their @id
record_sets = list(dataset.metadata.record_sets)
print(f"Available record sets ({len(record_sets)}):")
for i, recset in enumerate(record_sets):
    print(f"[{i}] @id: {recset.id}, name: {getattr(recset, 'name', 'N/A')}")

# For each record set, print the available fields (columns)
print("\nFields for each record set:")
for recset in record_sets:
    field_ids = []
    for field in recset.fields:
        field_ids.append((field.id, getattr(field, 'name', 'N/A')))
    print(f"Record Set @id: {recset.id}, Fields: {field_ids}")

## 3. Data Extraction
Load data from one or more record sets into pandas DataFrames for analysis. We'll use record set and field `@id`s as shown above.

_For this dataset, the primary record set appears to be the main survey table. We'll load all record sets for completeness, but focus on the main table for analysis._

In [ ]:
# Prepare list of record set @ids to extract
record_set_ids = [recset.id for recset in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records for record set {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Print columns for each record set (first 1-2)
for recset_id, df in dataframes.items():
    print(f"Record set {recset_id} columns: {df.columns.tolist()[:10]}{'...' if len(df.columns)>10 else ''}")

# Pick the main survey record set for demonstration.
# In this FAIR2 dataset, typically the main record set is the first one. Adjust record_set_id as needed by inspecting output above.

main_record_set_id = record_set_ids[0] if len(record_set_ids) else None
if main_record_set_id is not None and not dataframes[main_record_set_id].empty:
    print(f"\n--- Previewing first 5 rows of main record set {main_record_set_id} ---")
    display(dataframes[main_record_set_id].head())
else:
    print("No records found in main record set.")

## 4. Exploratory Data Analysis (EDA)
Apply basic data processing: filtering, normalization, grouping. We'll use field `@id`s to reference fields. We'll select a numeric field such as 'gad7_total' for examplar analysis, if available.

_Note:_ Adjust field `@id` as required; the list of field IDs is printed above.

In [ ]:
## Choose the main record set and a numeric field for illustration
import numpy as np

# For the FAIR2 Mental Health Survey, common numeric fields are e.g. 'gad7_total', 'phq9_total', 'psq_total'.
# The actual field @id can be found from field list above; let's programmatically pick one.
survey_df = None
numeric_field_id = None
categorical_group_field_id = None

if main_record_set_id is not None:
    df = dataframes[main_record_set_id]
    # Find a suitable numeric field for demonstration
    candidates = [col for col in df.columns if ('gad7' in col or 'phq9' in col or 'psq' in col) and 'total' in col]
    if candidates:
        numeric_field_id = candidates[0]

    # For grouping, pick a likely demographic
    possible_group_fields = [col for col in df.columns if 'gender' in col or 'sex' in col or 'education' in col]
    categorical_group_field_id = possible_group_fields[0] if possible_group_fields else None

    survey_df = df
    if numeric_field_id:
        # Remove non-numeric and missing values if any
        survey_df[numeric_field_id] = pd.to_numeric(survey_df[numeric_field_id], errors='coerce')
        threshold = survey_df[numeric_field_id].mean()  # Example: filter above mean
        filtered_df = survey_df[survey_df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean): {len(filtered_df)} entries")

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Sample of normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Grouping
        if categorical_group_field_id and categorical_group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(categorical_group_field_id)[numeric_field_id].mean().to_frame()
            print(f"\nGrouped mean {numeric_field_id} by {categorical_group_field_id}:")
            print(grouped_df)
    else:
        print("No suitable numeric field found for EDA.")
else:
    print('No main record set available.')

## 5. Visualization
Visualize distributions or relationships between fields, e.g., the distribution of a numeric score, or scores grouped by demographic attribute.

_Here, we use seaborn/matplotlib to plot histograms and group-wise boxplots._

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')

if survey_df is not None and numeric_field_id:
    plt.figure(figsize=(7,5))
    sns.histplot(survey_df[numeric_field_id].dropna(), bins=16, kde=True, color='deepskyblue')
    plt.xlabel(numeric_field_id)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.show()
    
    # Group-wise boxplot
    if categorical_group_field_id:
        plt.figure(figsize=(9,5))
        sns.boxplot(data=survey_df, x=categorical_group_field_id, y=numeric_field_id, palette='Set2')
        plt.ylabel(numeric_field_id)
        plt.xlabel(categorical_group_field_id)
        plt.title(f'{numeric_field_id} by {categorical_group_field_id}')
        plt.show()
else:
    print('Visualization skipped: no suitable survey DataFrame and numeric field.')

## 6. Conclusion
In this notebook, we've loaded a FAIR²-compliant mental health survey dataset from Kilifi County, Kenya using the `mlcroissant` library, explored its structure by referencing entities with their `@id`s, extracted data, and performed simple analyses on key indicators such as GAD-7/PHQ-9/PSQ scores.

This workflow illustrates AI-ready data processing on open, standardized survey data. For advanced analysis, you may integrate additional field-level metadata, explore correlations between scores, or customize the workflow to your research question.

_Please consult the dataset's documentation for a full data dictionary and ethical use guidelines._